# Ejemplo de uso 2: Importación y evaluación de un modelo ya entrenado

Este notebook ilustra cómo cargar un modelo ya entrenado (importándolo desde un checkpoint), y extraer a partir de él métricas de evaluación como Fréchet inception distance o Incepción score.

En primer lugar, importamos las funciones que vamos a necesitar.

In [1]:
from evaluate_metrics import compute_fid_and_iscore, cargar_modelo

<module 'diffusion_utilities' from 'c:\\Users\\cesar\\OneDrive\\Documentos\\Estudios\\Universidad\\3º carrera\\Segundo cuatri\\Aprendizaje Automatico III\\Proyecto IA generativa\\AA3_2024_2025_P02_Llorente_Capa_Villegas_Taillefer\\diffusion_utilities.py'>

Este paquete Python de generación de imágenes guarda "check_points" durante el entrenamiento de los modelos, lo que permite importar modelos ya entrenados para evaluarlos o generar imágenes sin necesidad alguna de entrenar el modelo, lo que supone un ahorro de tiempo considerable, sobretodo cuando se usan datasets muy grandes (como en este caso, CIFAR-10).

### Carga del modelo

Cargamos un modelo a partir de un checkpoint que tenemos guardado.

In [ ]:
checkpoint_path = "./checkpoints/checkpoint_cifar10_ve_exponential.pth"
model = cargar_modelo(checkpoint_path, schedule_type="exponential", diffusion_type="ve")

c:\Users\cesar\OneDrive\Documentos\Estudios\Universidad\3º carrera\Segundo cuatri\Aprendizaje Automatico III\Proyecto IA generativa\AA3_2024_2025_P02_Llorente_Capa_Villegas_Taillefer\evaluate_metrics.py:32: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please

Otra opción sería entrenar un nuevo modelo, pero esta vez vamos a seguir con el modelo importado a partir de un check_point.

In [ ]:
# from diffusion_functions import train_model
# model = train_model(n_epochs=30, dataset="cifar10", diffusion_type="ve", schedule_type = "exponential")

### Métricas de evaluación
Una vez tenemos el modelo entrenado, procedemos a evaluarlo. Para ello, usamos el módulo evaluate_metrics, donde la función compute_fid_and_iscore nos devuelve el FID (Fréchet inception distance) y el IS (Inception score), este último dividido en media y desviación típica.

In [5]:
fid_score, is_mean, is_std = compute_fid_and_iscore(model, dataset="cifar10")
print(f"[📊] FID: {fid_score:.2f} — Inception Score: {is_mean:.2f} ± {is_std:.2f}")

Files already downloaded and verified
[📊] FID: 5.91 — Inception Score: 1.82 ± 0.14


### Interpretación de los resultados

- El FID mide la “distancia” entre la distribución de características de las imágenes reales y las generadas. Un valor bajo del FID significa una calidad  alta, mientras que un valor alto demuestra una calidad pobre.
- El IS mide cuánto las imágenes generadas a) son “clasificables” (i.e. Inception da una clase fuerte, baja entropía condicional) y b) son diversas (la distribución marginal sobre clases es uniforme, alta entropía marginal). Un IS cercano a 1 representa una imagen sin contenido reconocible (ruido puro), un IS entre 5 y 8 representa una calidad moderada, y valores más altos indican una nitidez y diversidad muy altas en las imágenes. Es decir, cuanto más alto, mejor. La desviación estándar dice cuánta variación hay entre los distintos “splits” del cálculo: un IS con baja desviación (por ejemplo, ±0.1) es más estable.